
# Chapter 20 — Log File:



A log file is basically the application's diary.
It records what happened while your application was running, especially things that you may need to investigate later.

In [87]:
# Will make a log file in JSON format, which will keep data in this kind of format

In [88]:
# {
#   "timestamp": "2026-10-06T22:30:01",
#   "level": "INFO",
#   "event": "LLM_REQUEST",

#   "model": "openai/gpt-oss-20b",

#   "attempt": 1,

#   "status": "SUCCESS",

#   "input_tokens": 102,
#   "output_tokens": 1308,
#   "total_tokens": 1410,
#   "reasoning_tokens": 836,

#   "latency_seconds": 2.41,

#   "error": null
# }

      LLM Application
                    ↓
              Try Request
                    ↓
          ┌─────────┴─────────┐
          ↓                   ↓
       SUCCESS              ERROR
          ↓                   ↓
     Get metadata          Log error
          ↓                   ↓
     Token details         Retry
          ↓                   ↓
          └─────────┬─────────┘
                    ↓
              JSONL Log
                    ↓
          Request + Tokens
          + Latency + Error

In [89]:
from dotenv import load_dotenv
import os

load_dotenv()


os.environ["GROQ_API_KEY"] = "YOUR API"


In [90]:
# Setting up environment variable

def setup_environment():
  """ Loads environment variables and check for the required API key"""

  load_dotenv()
  if not os.getenv("GROQ_API_KEY"):
    raise ValueError("GROQ_API_KEY is not set")


In [91]:
# %pip install -U langchain langchain-core langchain-groq

In [92]:
#!pip install -U langchain-openai

In [93]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    model="openai/gpt-oss-20b",
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],

)

In [94]:
# Creating a JSON file name with date when the code we run

In [95]:
from datetime import datetime

file_name = f"llm_app_{datetime.now().strftime('%Y-%m-%d')}.jsonl"

print(file_name)

llm_app_2026-10-06.jsonl


In [96]:
import json
from datetime import datetime

In [97]:
import json
from datetime import datetime


def write_log(
    level,
    event,
    model,
    attempt,
    status,
    input_tokens=None,
    output_tokens=None,
    reasoning_tokens=None,
    total_tokens=None,
    latency_seconds=None,
    error=None
):

    log_data = {
        "timestamp": datetime.now().isoformat(),
        "level": level,
        "event": event,
        "model": model,
        "attempt": attempt,
        "status": status,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "reasoning_tokens": reasoning_tokens,
        "total_tokens": total_tokens,
        "latency_seconds": latency_seconds,
        "error": error
    }

    with open(file_name, "a") as file:
        file.write(json.dumps(log_data) + "\n")

In [98]:
write_log(
    level="INFO",
    event="TEST",
    model="openai/gpt-oss-20b",
    attempt=1,
    status="SUCCESS"
)

In [104]:
import time


def ask_llm(prompt):

    start_time = time.time()

    try:

        response = llm.invoke(prompt)

        latency = time.time() - start_time

        usage = response.usage_metadata

        write_log(
            level="INFO",
            event="LLM_REQUEST",
            model="openai/gpt-oss-20b",
            attempt=1,
            status="SUCCESS",
            input_tokens=usage.get("input_tokens"),
            output_tokens=usage.get("output_tokens"),
            reasoning_tokens=usage.get(
                "output_token_details", {}
            ).get("reasoning"),
            total_tokens=usage.get("total_tokens"),
            latency_seconds=round(latency, 3)
        )

        return response.content

    except Exception as e:

        latency = time.time() - start_time

        write_log(
            level="ERROR",
            event="LLM_REQUEST",
            model="openai/gpt-oss-20b",
            attempt=1,
            status="FAILED",
            latency_seconds=round(latency, 3),
            error=str(e)
        )

        return None

In [105]:
answer = ask_llm(
    "Explain RAG in simple words."
)

print(answer)

**RAG (Retrieval‑Augmented Generation) in plain terms**

Imagine you’re writing a story but you don’t know all the facts. Instead of just guessing, you can **look up** the exact facts you need from a big library, then write the story using those facts. That’s what a Retrieval‑Augmented Generation model does:

| Step | What happens | Why it helps |
|------|--------------|--------------|
| **1. Ask the question** | The user gives a prompt or question. | The model gets a clear target. |
| **2. Retrieve** | The model searches a “knowledge base” (a set of documents, a database, or the web) for passages that are likely to contain the answer. | It brings in fresh, up‑to‑date facts that the model itself might not have memorized. |
| **3. Fuse** | The retrieved passages are fed back into the model along with the question. | The model now has the right pieces of information right in front of it. |
| **4. Generate** | The model writes the final answer, weaving the retrieved facts into natural lan